In [ ]:
from pydantic import BaseModel
from typing import List
from langchain_core.output_parsers import PydanticOutputParser
from sentence_transformers import CrossEncoder

model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")
class QueryList(BaseModel):
    queries: List[str]

output_parser = PydanticOutputParser(pydantic_object=QueryList)

In [2]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT_TEMPLATE = """
Give the message {input}, return the query rephrased in 5 ways, so that rag retrieval for the query isn't missed. Output just the generated messages. Return the result in {output} format.
"""

query_prompt_template = ChatPromptTemplate(
    messages=[
        PROMPT_TEMPLATE
    ],
    partial_variables={"output": output_parser.get_format_instructions()}
)

In [3]:
from langchain_openai import ChatOpenAI

grok_api_key = ""

chat_model = ChatOpenAI(
    api_key= grok_api_key ,
    base_url="https://api.groq.com/openai/v1",
    model="llama-3.3-70b-versatile"   # Example model
)


query_generator_chain = query_prompt_template | chat_model | output_parser

def generate_queries_chunks(input_query):
    response_for_multiple_queries = query_generator_chain.invoke({'input' : input_query}).queries
    all_matching_chunks = list(set([ k  for i in  response_for_multiple_queries for j in hybrid_retriever(i)['documents'] for k in j]))
    pairs = [[input_query, doc] for doc in all_matching_chunks]
    scores = model.predict(pairs)
    ranked_docs = sorted(zip(all_matching_chunks, scores), key=lambda x: x[1], reverse=True)
    return (ranked_docs[:5])

In [5]:
import chromadb
from chromadb import Schema, SparseVectorIndexConfig, VectorIndexConfig, K
from chromadb.utils.embedding_functions import ChromaBm25EmbeddingFunction, SentenceTransformerEmbeddingFunction


hf_embedding = SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-small-en-v1.5"
)
client = chromadb.CloudClient(
  api_key="",
  tenant="9cb6a35c-76c7-48f0-b1cf-187e130f859b",
  database='chroma_vector_db'
)

schema = Schema()
    
schema = schema.create_index(
  key='sparse_vector_key',    
  config=SparseVectorIndexConfig(
    source_key=K.DOCUMENT,
    bm25=True,
    embedding_function=ChromaBm25EmbeddingFunction(
        k=1.2,
        b=0.75,
        avg_doc_length=256.0,
        token_max_length=40
    ),
  )
)



schema = schema.create_index(
    config=VectorIndexConfig(
        space="cosine",
         embedding_function=hf_embedding
    )
)

collection = client.get_or_create_collection(
  name="multi_query_schema",
  schema=schema,
)

collection.count()

0

In [6]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader('data', glob="*.txt", show_progress=True, loader_cls=TextLoader)

kb_docs = loader.load()

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 31.46it/s]


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = text_splitter.split_documents(kb_docs)

In [8]:
import uuid

# Free tier in chroma supports 300 docs to be inserted at a time
BATCH_SIZE = 250

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]

    collection.add(
        documents=[chunk.page_content for chunk in batch],
        metadatas=[chunk.metadata for chunk in batch],
        ids=[str(uuid.uuid4()) for _ in batch]
    )

    print(f"Inserted {min(i + BATCH_SIZE, len(chunks))}/{len(chunks)}")

Inserted 181/181


In [9]:
from chromadb import Knn, K
from chromadb import Search
from chromadb import Rrf

def hybrid_retriever(query):
    # Sparse embeddings
    sparse_rank = Knn(
      query=query,  # Text query for sparse embeddings
      key="sparse_vector_key",  # Metadata field for sparse vectors
      return_rank=True,
      limit=20                 # Only the 20 nearest documents get scored (default limit 16)
    )
    
    # Dense semantic embeddings
    dense_rank = Knn(
      query=query,  # Text query for dense embeddings
      key="#embedding",          # Default embedding field
      return_rank=True,
      limit=20                 # Only the 20 nearest documents get scored (default limit 16)

    )
    
    # Combine with RRF
    hybrid_rank = Rrf(
      ranks=[dense_rank, sparse_rank],
      weights=[0.7, 0.3],  # 70% semantic, 30% keyword
      k=60                 # smoothing parameter - higher values reduce emphasis on top ranks
    )
    
    # Use in search
    hybrid_search = (Search()
      .rank(hybrid_rank)
      .limit(5)
      .select(K.DOCUMENT, K.SCORE, K.METADATA)
    )
    
    hybrid_results = collection.search(hybrid_search)
    
    return hybrid_results

In [11]:
def format_docs(docs):
    context = ""
    for row in docs:
        context += row[0]
    return context

from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Wrap the functions in RunnableWrapper
hybrid_retriever_runnable = RunnableLambda(lambda x: generate_queries_chunks(x))
format_docs_runnable = RunnableLambda(lambda x: format_docs(x))


MAIN_PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}
Answer the question based on the above context: {question}.
Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

main_prompt_template = ChatPromptTemplate(
    messages=[
        MAIN_PROMPT_TEMPLATE
    ]
)



from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

main_generator_chain = main_prompt_template | chat_model | parser

main_rag_chain = {
    "context": hybrid_retriever_runnable | format_docs_runnable, 
    "question": RunnablePassthrough()
} | main_generator_chain

Rerun from here for each new query!!


In [76]:
input_query = input('Enter the query regarding Alizhmers you want to look up for.') 
main_rag_chain.invoke(input_query)

Enter the query regarding Alizhmers you want to look up for. What impact does Alzheimer’s disease have on caregivers?


"Caregivers can supply important information on daily living abilities and on the decrease in the person's mental function. A caregiver's viewpoint is particularly important, since a person with Alzheimer's disease is commonly unaware of their deficits. Many times, families have difficulties in the detection of initial symptoms, which implies that caregivers play a crucial role in detection and assessment. Caregivers are also involved in the assessment process through interviews with family members, highlighting their significant role in the diagnosis and understanding of Alzheimer's disease."

In [12]:
main_rag_chain.invoke('What are amyloid plaques?')

'AÎ² plaques are dense, mostly insoluble deposits of amyloid beta peptide and cellular material outside and around neurons.'